In [53]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

In [54]:
# Load dataset
digits = load_digits()
X, y = digits.data, digits.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [58]:
# Define separate estimators for the selector and the final classifier
selector = SVC(kernel='linear', C=1.0, probability=True)
clf = SVC(C=1.0, gamma=0.1, probability=True)

# Define the feature selector - Recursive Feature Elimination
rfecv = RFECV(
    estimator=selector,
    step=1,
    cv=StratifiedKFold(10),
    scoring='roc_auc_ovr'
)

# Create a pipeline
pipeline = Pipeline([
    ('feature_selection', rfecv),
    ('classification', clf)
])

In [59]:
# Define the SVC parameter grid
param_grid = {
    'classification__C': [0.1, 1.0, 10.0],
    'classification__gamma': ['scale', 0.01, 0.1],
    'classification__kernel': ['rbf', 'linear'],
}

In [60]:
# Define the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(10),
    scoring='roc_auc_ovr',
    n_jobs=-1,
    error_score='raise'
)

# Fit the model
grid_search.fit(X_train, y_train)

# Get the best parameters and score
print('Best parameters found:', grid_search.best_params_)
print('Best cross-validation score:', grid_search.best_score_)

KeyboardInterrupt: 

In [ ]:
# Predict on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)

# Evaluate the model
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print('ROC AUC Score on test data:', roc_auc)

NotFittedError: This GridSearchCV instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.